# 소주제 D. 설비 고장 유형 예측

> **AI 기반 스마트 팩토리 품질관리 시스템 — LSTM / BERT / DNN 비교**
>
> K-Digital Training 딥러닝 12기 미니프로젝트

---

## 핵심 아이디어
센서 수치 데이터를 **자연어 텍스트로 변환**한 뒤 NLP 모델(LSTM/BERT)로 고장 유형을 분류하고,
수치 기반 DNN 베이스라인과 비교합니다.

| 단계 | 내용 |
|------|------|
| 1 | 환경 설정 |
| 2 | 데이터 로드 및 EDA |
| 3 | 피처 엔지니어링 + 수치→텍스트 변환 |
| 4 | LSTM 모델 (Bi-LSTM + Attention) |
| 5 | BERT 모델 (DistilBERT) |
| 6 | DNN 베이스라인 |
| 7 | 3모델 비교 및 혼동행렬 |
| 8 | 워드클라우드 + 결과 저장 |

## 1. 환경 설정

In [ ]:
# !pip install -q transformers datasets wordcloud

import os
import json
import random
import warnings
import math
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, TensorDataset

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix, classification_report
)

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer
)
from datasets import Dataset as HFDataset

warnings.filterwarnings('ignore')
matplotlib.rc('font', family='AppleGothic')
matplotlib.rcParams['axes.unicode_minus'] = False

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

NUM_WORKERS = 0  # 노트북 pickle 오류 방지
NUM_CLASSES = 6

if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f'GPU: {torch.cuda.get_device_name(0)}')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
    print('Apple Silicon MPS')
else:
    device = torch.device('cpu')
    print('CPU 사용')

print(f'PyTorch: {torch.__version__}')
print(f'Device: {device}')

In [ ]:
# --- 경로 ---
BASE_DIR   = Path('../')
DATA_PATH  = Path('../../data/predictive_maintenance.csv')
MODEL_DIR  = BASE_DIR / 'models'
RESULT_DIR = BASE_DIR / 'results'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

# LABEL_NAMES는 Cell 3 (LabelEncoder 후)에서 동적 생성
               'Power Failure', 'Random Failures', 'Tool Wear Failure']


## 2. 데이터 로드 및 EDA

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
print(f'결측치: {df.isnull().sum().sum()}')
print(f'\nFailure Type 분포:')
print(df['Failure Type'].value_counts())
print(f'\nType 분포:')
print(df['Type'].value_counts())
df.head()

In [ ]:
# --- EDA 시각화 ---
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Failure Type 분포
ft_counts = df['Failure Type'].value_counts()
colors = sns.color_palette('Set2', len(ft_counts))
bars = axes[0, 0].barh(range(len(ft_counts)), ft_counts.values, color=colors)
axes[0, 0].set_yticks(range(len(ft_counts)))
axes[0, 0].set_yticklabels(ft_counts.index, fontsize=9)
axes[0, 0].set_xlabel('건수')
axes[0, 0].set_title('고장 유형별 분포')
axes[0, 0].bar_label(bars, padding=3, fontsize=8)

# 수치 피처 히스토그램
num_cols = ['Air temperature [K]', 'Process temperature [K]',
            'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']
for idx, col in enumerate(num_cols):
    row, col_idx = (idx + 1) // 3, (idx + 1) % 3
    axes[row, col_idx].hist(df[col], bins=30, color='#3498DB', edgecolor='white', alpha=0.8)
    axes[row, col_idx].set_title(col, fontsize=10)
    axes[row, col_idx].set_ylabel('빈도')

plt.suptitle('EDA: 데이터 분포 개요', fontsize=14)
plt.tight_layout()
plt.savefig(RESULT_DIR / '01_eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. 피처 엔지니어링 + 수치→텍스트 변환

In [ ]:
def row_to_text(row):
    """센서 수치 → 자연어 텍스트 변환"""
    air_t = row['Air temperature [K]']
    proc_t = row['Process temperature [K]']
    rpm = row['Rotational speed [rpm]']
    torque = row['Torque [Nm]']
    wear = row['Tool wear [min]']
    mtype = row['Type']

    # 범주화
    air_cat = 'low' if air_t < 298 else ('high' if air_t >= 302 else 'normal')
    proc_cat = 'low' if proc_t < 308 else ('high' if proc_t >= 311 else 'normal')
    rpm_cat = 'low' if rpm < 1300 else ('high' if rpm >= 1600 else 'normal')
    torque_cat = 'low' if torque < 30 else ('high' if torque >= 50 else 'normal')
    wear_cat = 'minimal' if wear < 50 else ('severe' if wear >= 150 else 'moderate')

    # 파생 피처
    temp_diff = abs(proc_t - air_t)
    power = torque * rpm * 2 * math.pi / 60

    text = (
        f"Type {mtype} machine. "
        f"Air temperature {air_t:.1f}K {air_cat}. "
        f"Process temperature {proc_t:.1f}K {proc_cat}. "
        f"Rotational speed {rpm}rpm {rpm_cat}. "
        f"Torque {torque:.1f}Nm {torque_cat}. "
        f"Tool wear {wear}min {wear_cat}. "
        f"Temperature difference {temp_diff:.1f}K. "
        f"Estimated power {power:.0f}W."
    )
    return text


# 텍스트 생성
df['text'] = df.apply(row_to_text, axis=1)
print('샘플 텍스트:')
print(df['text'].iloc[0])

# 라벨 인코딩
le = LabelEncoder()
df['label'] = le.fit_transform(df['Failure Type'])
LABEL_NAMES = list(le.classes_)
# 혼동행렬 축 라벨 (LabelEncoder 순서에 맞춤)
_short_map = {
    "Heat Dissipation Failure": "Heat", "No Failure": "No Fail",
    "Overstrain Failure": "Overstrain", "Power Failure": "Power",
    "Random Failures": "Random", "Tool Wear Failure": "Tool Wear",
}
SHORT_NAMES = [_short_map[c] for c in LABEL_NAMES]
print(f"SHORT_NAMES 순서: {SHORT_NAMES}")
print(f'\n라벨 매핑: {dict(zip(le.classes_, le.transform(le.classes_)))}')

# 클래스 가중치
class_counts = df['label'].value_counts().sort_index().values.astype(float)
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum() * NUM_CLASSES
class_weights_tensor = torch.FloatTensor(class_weights).to(device)
print(f'클래스 가중치: {class_weights.round(3)}')

# 데이터 분할 (70/15/15 stratified)
train_df, temp_df = train_test_split(df, test_size=0.3, random_state=SEED, stratify=df['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=SEED, stratify=temp_df['label'])

print(f'\nTrain: {len(train_df):,}, Val: {len(val_df):,}, Test: {len(test_df):,}')

In [ ]:
# --- 피처별 고장 유형 boxplot ---
num_cols = ['Air temperature [K]', 'Process temperature [K]',
            'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for idx, col in enumerate(num_cols):
    ax = axes[idx // 3, idx % 3]
    # 고장 유형별 boxplot (No Failure 제외)
    fail_df = df[df['Failure Type'] != 'No Failure']
    sns.boxplot(data=fail_df, x='Failure Type', y=col, ax=ax, palette='Set2')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right', fontsize=8)
    ax.set_title(col, fontsize=10)

axes[1, 2].axis('off')
plt.suptitle('고장 유형별 수치 피처 분포 (No Failure 제외)', fontsize=14)
plt.tight_layout()
plt.savefig(RESULT_DIR / '02_feature_by_failure.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. LSTM 모델 (Bi-LSTM + Attention)

In [ ]:
# --- Vocabulary & Dataset ---
MAX_LEN = 128

def build_vocab(texts, min_freq=1):
    counter = Counter()
    for text in texts:
        counter.update(text.lower().split())
    vocab = {'<PAD>': 0, '<UNK>': 1}
    for word, cnt in counter.items():
        if cnt >= min_freq:
            vocab[word] = len(vocab)
    return vocab

def text_to_indices(text, vocab, max_len=MAX_LEN):
    tokens = text.lower().split()[:max_len]
    indices = [vocab.get(t, 1) for t in tokens]  # UNK=1
    indices += [0] * (max_len - len(indices))      # PAD=0
    return indices

class MaintenanceDataset(Dataset):
    def __init__(self, texts, labels, vocab):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        indices = text_to_indices(self.texts[idx], self.vocab)
        return torch.tensor(indices, dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.long)


vocab = build_vocab(train_df['text'].tolist())
print(f'Vocabulary 크기: {len(vocab)}')

train_ds = MaintenanceDataset(train_df['text'].tolist(), train_df['label'].tolist(), vocab)
val_ds   = MaintenanceDataset(val_df['text'].tolist(), val_df['label'].tolist(), vocab)
test_ds  = MaintenanceDataset(test_df['text'].tolist(), test_df['label'].tolist(), vocab)

train_dl = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=NUM_WORKERS)
val_dl   = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=NUM_WORKERS)
test_dl  = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=NUM_WORKERS)

In [ ]:
class LSTMClassifier(nn.Module):
    """Bi-LSTM + Attention"""
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=256,
                 num_layers=2, num_classes=NUM_CLASSES, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers,
                           batch_first=True, bidirectional=True, dropout=dropout)
        self.attention = nn.Linear(hidden_dim * 2, 1)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        emb = self.embedding(x)
        out, _ = self.lstm(emb)
        attn_weights = torch.softmax(self.attention(out).squeeze(-1), dim=1)
        context = torch.bmm(attn_weights.unsqueeze(1), out).squeeze(1)
        return self.classifier(context)


lstm_model = LSTMClassifier(len(vocab)).to(device)
total_p = sum(p.numel() for p in lstm_model.parameters())
print(f'LSTM 파라미터: {total_p:,}')

In [ ]:
# --- LSTM 학습 ---
LSTM_EPOCHS = 20
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = optim.Adam(lstm_model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)

lstm_history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_acc = 0.0

for epoch in range(1, LSTM_EPOCHS + 1):
    # Train
    lstm_model.train()
    tl, tc, tn = 0, 0, 0
    for x, y in train_dl:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = lstm_model(x)
        loss = criterion(out, y)
        loss.backward()
        nn.utils.clip_grad_norm_(lstm_model.parameters(), 1.0)
        optimizer.step()
        tl += loss.item() * x.size(0)
        tc += out.argmax(1).eq(y).sum().item()
        tn += x.size(0)

    # Val
    lstm_model.eval()
    vl, vc, vn = 0, 0, 0
    with torch.no_grad():
        for x, y in val_dl:
            x, y = x.to(device), y.to(device)
            out = lstm_model(x)
            loss = criterion(out, y)
            vl += loss.item() * x.size(0)
            vc += out.argmax(1).eq(y).sum().item()
            vn += x.size(0)

    t_loss, t_acc = tl/tn, tc/tn
    v_loss, v_acc = vl/vn, vc/vn
    scheduler.step(v_acc)

    lstm_history['train_loss'].append(t_loss)
    lstm_history['val_loss'].append(v_loss)
    lstm_history['train_acc'].append(t_acc)
    lstm_history['val_acc'].append(v_acc)

    improved = ''
    if v_acc > best_val_acc:
        best_val_acc = v_acc
        torch.save(lstm_model.state_dict(), MODEL_DIR / 'best_lstm.pth')
        improved = ' ★'

    print(f'  Epoch {epoch:2d}/{LSTM_EPOCHS} | '
          f'Loss: {t_loss:.4f}/{v_loss:.4f} | Acc: {t_acc:.4f}/{v_acc:.4f}{improved}')

lstm_model.load_state_dict(torch.load(MODEL_DIR / 'best_lstm.pth', weights_only=True))
print(f'Best Val Acc: {best_val_acc:.4f}')

# Training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ep = range(1, len(lstm_history['train_loss']) + 1)
axes[0].plot(ep, lstm_history['train_loss'], 'b--', label='Train')
axes[0].plot(ep, lstm_history['val_loss'], 'r-', label='Val')
axes[0].set_title('LSTM Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(ep, lstm_history['train_acc'], 'b--', label='Train')
axes[1].plot(ep, lstm_history['val_acc'], 'r-', label='Val')
axes[1].set_title('LSTM Accuracy'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(RESULT_DIR / '03_lstm_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. BERT 모델 (DistilBERT)

In [ ]:
# --- DistilBERT 학습 ---
BERT_MODEL = 'distilbert-base-uncased'
BERT_EPOCHS = 5
BERT_MAX_LEN = 128

tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)

def tokenize_fn(examples):
    return tokenizer(examples['text'], padding='max_length',
                     truncation=True, max_length=BERT_MAX_LEN)

# HuggingFace Dataset 변환
train_hf = HFDataset.from_pandas(train_df[['text', 'label']].reset_index(drop=True))
val_hf   = HFDataset.from_pandas(val_df[['text', 'label']].reset_index(drop=True))
test_hf  = HFDataset.from_pandas(test_df[['text', 'label']].reset_index(drop=True))

train_tok = train_hf.map(tokenize_fn, batched=True)
val_tok   = val_hf.map(tokenize_fn, batched=True)
test_tok  = test_hf.map(tokenize_fn, batched=True)

train_tok.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
val_tok.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
test_tok.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

# 모델
bert_model = AutoModelForSequenceClassification.from_pretrained(
    BERT_MODEL, num_labels=NUM_CLASSES
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1_macro': f1_score(labels, preds, average='macro'),
    }

training_args = TrainingArguments(
    output_dir=str(MODEL_DIR / 'bert_output'),
    num_train_epochs=BERT_EPOCHS,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    logging_steps=50,
    report_to='none',
    seed=SEED,
)

trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    compute_metrics=compute_metrics,
)

print('DistilBERT 학습 시작...')
trainer.train()
print('DistilBERT 학습 완료!')

## 6. DNN 베이스라인

In [ ]:
# --- 수치 피처 준비 ---
def prepare_numeric_features(dataframe):
    feat = dataframe[['Air temperature [K]', 'Process temperature [K]',
                      'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']].copy()
    feat['temp_diff'] = feat['Process temperature [K]'] - feat['Air temperature [K]']
    feat['power'] = feat['Torque [Nm]'] * feat['Rotational speed [rpm]'] * 2 * math.pi / 60
    feat['type_encoded'] = dataframe['Type'].map({'L': 0, 'M': 1, 'H': 2})
    return feat.values

X_train_num = prepare_numeric_features(train_df)
X_val_num   = prepare_numeric_features(val_df)
X_test_num  = prepare_numeric_features(test_df)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_num)
X_val_scaled   = scaler.transform(X_val_num)
X_test_scaled  = scaler.transform(X_test_num)

y_train = train_df['label'].values
y_val   = val_df['label'].values
y_test  = test_df['label'].values

train_tensor = TensorDataset(torch.FloatTensor(X_train_scaled), torch.LongTensor(y_train))
val_tensor   = TensorDataset(torch.FloatTensor(X_val_scaled), torch.LongTensor(y_val))

dnn_train_dl = DataLoader(train_tensor, batch_size=64, shuffle=True, num_workers=NUM_WORKERS)
dnn_val_dl   = DataLoader(val_tensor, batch_size=64, shuffle=False, num_workers=NUM_WORKERS)


class DNNClassifier(nn.Module):
    def __init__(self, in_features=8, num_classes=NUM_CLASSES):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, num_classes),
        )

    def forward(self, x):
        return self.net(x)


# DNN 학습
DNN_EPOCHS = 30
dnn_model = DNNClassifier().to(device)
dnn_criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
dnn_optimizer = optim.Adam(dnn_model.parameters(), lr=1e-3)

best_dnn_acc = 0.0
for epoch in range(1, DNN_EPOCHS + 1):
    dnn_model.train()
    for x, y in dnn_train_dl:
        x, y = x.to(device), y.to(device)
        dnn_optimizer.zero_grad()
        loss = dnn_criterion(dnn_model(x), y)
        loss.backward()
        dnn_optimizer.step()

    dnn_model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in dnn_val_dl:
            x, y = x.to(device), y.to(device)
            correct += dnn_model(x).argmax(1).eq(y).sum().item()
            total += y.size(0)
    acc = correct / total
    if acc > best_dnn_acc:
        best_dnn_acc = acc
        torch.save(dnn_model.state_dict(), MODEL_DIR / 'best_dnn.pth')

    if epoch % 10 == 0 or epoch == 1:
        print(f'  DNN Epoch {epoch:2d}/{DNN_EPOCHS} | Val Acc: {acc:.4f}')

dnn_model.load_state_dict(torch.load(MODEL_DIR / 'best_dnn.pth', weights_only=True))
print(f'Best DNN Val Acc: {best_dnn_acc:.4f}')

## 7. 3모델 비교 및 혼동행렬

In [ ]:
# --- Test 예측 ---
def predict_lstm(model, loader):
    model.eval()
    preds = []
    with torch.no_grad():
        for x, _ in loader:
            x = x.to(device)
            preds.extend(model(x).argmax(1).cpu().numpy())
    return np.array(preds)

def predict_bert(trainer_obj, dataset):
    output = trainer_obj.predict(dataset)
    return np.argmax(output.predictions, axis=-1)

def predict_dnn(model, X_scaled):
    model.eval()
    with torch.no_grad():
        x = torch.FloatTensor(X_scaled).to(device)
        return model(x).argmax(1).cpu().numpy()


lstm_preds = predict_lstm(lstm_model, test_dl)
bert_preds = predict_bert(trainer, test_tok)
dnn_preds  = predict_dnn(dnn_model, X_test_scaled)

# 비교 테이블
results = {}
for name, preds in [('DNN (수치)', dnn_preds), ('LSTM (텍스트)', lstm_preds), ('BERT (텍스트)', bert_preds)]:
    results[name] = {
        'accuracy': accuracy_score(y_test, preds),
        'f1_macro': f1_score(y_test, preds, average='macro'),
        'f1_weighted': f1_score(y_test, preds, average='weighted'),
    }

print(f'{"모델":>18} | {"Accuracy":>9} | {"F1(macro)":>10} | {"F1(weighted)":>12}')
print('-' * 58)
for name, m in results.items():
    print(f'{name:>18} | {m["accuracy"]:>8.4f} | {m["f1_macro"]:>9.4f} | {m["f1_weighted"]:>11.4f}')

In [ ]:
# --- 혼동행렬 ---
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for idx, (name, preds) in enumerate([('DNN', dnn_preds), ('LSTM', lstm_preds), ('BERT', bert_preds)]):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=SHORT_NAMES, yticklabels=SHORT_NAMES,
                cbar=False, annot_kws={'size': 9})
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')
    axes[idx].set_title(f'{name}', fontsize=12, fontweight='bold')
    axes[idx].tick_params(axis='both', labelsize=8)

plt.suptitle('3모델 혼동행렬 비교 (Test Set)', fontsize=14)
plt.tight_layout()
plt.savefig(RESULT_DIR / '04_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

# 비교 막대 그래프
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(results))
w = 0.25
model_names = list(results.keys())
for i, (metric, label) in enumerate([('accuracy', 'Accuracy'), ('f1_macro', 'F1(macro)'), ('f1_weighted', 'F1(weighted)')]):
    vals = [results[m][metric] for m in model_names]
    bars = ax.bar(x + i * w, vals, w, label=label)
    ax.bar_label(bars, fmt='%.3f', padding=2, fontsize=8)
ax.set_xticks(x + w)
ax.set_xticklabels(model_names)
ax.set_ylim(0, 1.1)
ax.set_title('모델 성능 비교')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(RESULT_DIR / '05_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. 워드클라우드 + 결과 저장

In [ ]:
# --- 고장 유형별 워드클라우드 ---
fail_types = [ft for ft in LABEL_NAMES if ft != 'No Failure']
n = len(fail_types)
cols = 3
rows = (n + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(15, 5 * rows))
axes = axes.flatten() if rows > 1 else [axes] if cols == 1 else axes.flatten()

for idx, ft in enumerate(fail_types):
    texts = df[df['Failure Type'] == ft]['text'].tolist()
    combined = ' '.join(texts)
    wc = WordCloud(width=400, height=300, max_words=50,
                   colormap='Reds', background_color='white').generate(combined)
    axes[idx].imshow(wc, interpolation='bilinear')
    axes[idx].set_title(ft, fontsize=11, fontweight='bold')
    axes[idx].axis('off')

for i in range(len(fail_types), len(axes)):
    axes[i].axis('off')

plt.suptitle('고장 유형별 워드클라우드', fontsize=14)
plt.tight_layout()
plt.savefig(RESULT_DIR / '06_wordclouds.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- JSON 저장 ---
best_model_name = max(results, key=lambda m: results[m]['f1_macro'])

summary = {
    'project': '소주제 D — 설비 고장 유형 예측',
    'dataset': 'Machine Predictive Maintenance (10,000건)',
    'num_classes': NUM_CLASSES,
    'classes': LABEL_NAMES,
    'approach': '수치→텍스트 변환 후 NLP 모델 적용',
    'data_split': {'train': len(train_df), 'val': len(val_df), 'test': len(test_df)},
    'model_comparison': {
        name: {k: round(v, 4) for k, v in m.items()}
        for name, m in results.items()
    },
    'best_model': best_model_name,
    'seed': SEED,
}

with open(RESULT_DIR / '07_final_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

# 최종 요약
print('=' * 60)
print(' 소주제 D — 설비 고장 유형 예측 | 최종 결과')
print('=' * 60)
print(f'\n{"모델":>18} | {"Accuracy":>9} | {"F1(macro)":>10} | {"F1(weighted)":>12}')
print('-' * 58)
for name, m in results.items():
    marker = ' ★' if name == best_model_name else ''
    print(f'{name:>18} | {m["accuracy"]:>8.2%} | {m["f1_macro"]:>9.2%} | {m["f1_weighted"]:>11.2%}{marker}')

print(f'\n★ 최고 모델 (F1 macro): {best_model_name}')
print(f'\n저장된 파일:')
for f in sorted(RESULT_DIR.glob('*')):
    print(f'  → results/{f.name}')
for f in sorted(MODEL_DIR.glob('*.pth')):
    print(f'  → models/{f.name}')